# Gene Cartoon

Generates a gene structure cartoon showing exon positions with compressed introns and UTRs.
If a library targets file is provided, a library amplicon track is added below the exon track.

**Required data:** Exon coordinates are fetched automatically from the Ensembl REST API (requires internet access). Optionally, a targets TSV file (`editstart`/`editstop` columns) can be provided to show library amplicons.

**Alternative data source:** If you have a cartoon metadata Excel file (`exon_coords`, `metadata`, and optionally `lib_coords` sheets), you can load from that instead — see the commented block in the data loading cell.

**Required packages:** `sge-utils` (SimpleSGEViz) installed in the active environment.

**Saving:** PNG/SVG require `vl-convert-python` (`pip install vl-convert-python`). Change the extension to `.html` for interactive output (no extra package needed).

In [ ]:
import pandas as pd
from pathlib import Path
from sgeviz import io
from sgeviz.figures import gene_cartoon

In [ ]:
# --- Configuration ---
gene = "GENE"                   # gene symbol (used in Ensembl lookup and output filename)
transcript_id = None            # Ensembl transcript ID (e.g. 'ENST00000357654'); None = auto-select canonical
assembly = "GRCh38"             # genome assembly for Ensembl lookup: 'GRCh38' or 'GRCh37'
targets_path = None             # path to targets TSV (editstart/editstop columns); None for exon-only cartoon
domains_path = None             # path to domain annotation file (CSV or Excel); None to omit
output_path = f"{gene}_gene_cartoon.png"  # change to .html for interactive output, or .svg

# --- Plot customization (optional) ---
width = 800         # data-area width in pixels
exon_color = None   # override exon color (e.g. '#2E86C1'); None uses Ensembl/metadata default
lib_color = None    # override library amplicon color (e.g. '#888888'); only applies with targets_path

In [ ]:
# --- Load exon coordinates ---
# Option A (default): fetch from Ensembl REST API
exon_df, _, meta_df = io.fetch_exon_coords(gene, transcript_id=transcript_id, assembly=assembly)

# Option B: load from a cartoon metadata Excel file instead
# cartoon_path = "GENE_cartoon.xlsx"
# xl = pd.ExcelFile(cartoon_path)
# exon_df = xl.parse("exon_coords")
# meta_df = xl.parse("metadata")

# --- Load library amplicons (optional) ---
lib_df = None
if targets_path is not None:
    targets_df = pd.read_csv(targets_path, sep="\t")
    lib_df = targets_df[["editstart", "editstop"]].rename(
        columns={"editstart": "start", "editstop": "end"}
    )
    print(f"Loaded {len(lib_df)} library amplicons from {targets_path}")

In [ ]:
# --- Generate plot ---
domains = Path(domains_path) if domains_path else None

if lib_df is not None and not lib_df.empty:
    chart = gene_cartoon.make_library_cartoon(
        exon_df, lib_df, meta_df,
        width=width,
        exon_color=exon_color,
        lib_color=lib_color,
        domains_path=domains,
    )
else:
    chart = gene_cartoon.make_exon_cartoon(
        exon_df, meta_df,
        width=width,
        exon_color=exon_color,
        domains_path=domains,
    )
chart

In [ ]:
# --- Save ---
chart.save(output_path)
print(f"Saved: {output_path}")